# Preprocessing Counters Utilities Demo

Demostración interactiva de las utilidades de conteo y diagnóstico métrico de calidad de datos implementadas en `_preprocessing_counters.py` y expuestas por `ds_utils.preprocessing.preprocessing_utilities`.

Funciones cubiertas en este notebook:
- `count_missing_timestamps`
- `count_edge_null_rows`
- `count_rows_by_time`
- `count_constant_columns`
- `count_cells_with_missing`
- `count_rows_with_missing`
- `count_columns_with_missing`
- `count_missing_cells_by_column`

## Imports

In [2]:
# RECARGA AUTOMÁTICA DE MÓDULOS EXTERNOS
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from ds_utils.preprocessing.preprocessing_utilities import (
    count_missing_timestamps,
    count_edge_null_rows,
    count_rows_by_time,
    count_constant_columns,
    count_cells_with_missing,
    count_rows_with_missing,
    count_columns_with_missing,
    count_missing_cells_by_column,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## count_missing_timestamps

`count_missing_timestamps` compara la serie temporal observada contra la secuencia completa esperada según `freq` y devuelve el número de marcas faltantes y su `DatetimeIndex`.

In [3]:
# Serie con huecos en las 02:00 y las 04:00
df_ts = pd.DataFrame({
    "timestamp": ["2024-01-01 00:00:00", "2024-01-01 01:00:00", "2024-01-01 03:00:00", "2024-01-01 05:00:00"],
    "val": [10, 15, 20, 25]
})

n_missing_ts, missing_idx = count_missing_timestamps(df=df_ts, col_time="timestamp", freq="1h")

print(f"Timestamps ausentes detectados: {n_missing_ts}")
display(missing_idx)

Timestamps ausentes detectados: 2


DatetimeIndex(['2024-01-01 02:00:00', '2024-01-01 04:00:00'], dtype='datetime64[us]', freq=None)

## count_edge_null_rows

`count_edge_null_rows` contabiliza cuántas filas consecutivas con nulos existen en los bordes inicial y final (`n_start`, `n_end`) bajo modo `'any'` o `'all'`.

In [4]:
df_edges = pd.DataFrame({
    "a": [np.nan, 10.0, 20.0, 30.0, np.nan],
    "b": [np.nan, np.nan, 2.5, 3.5, np.nan]
})

start_any, end_any = count_edge_null_rows(df=df_edges, mode="any")
start_all, end_all = count_edge_null_rows(df=df_edges, mode="all")

print(f"Bordes con algún nulo (any): inicio={start_any}, fin={end_any}")
print(f"Bordes completamente nulos (all): inicio={start_all}, fin={end_all}")

Bordes con algún nulo (any): inicio=2, fin=1
Bordes completamente nulos (all): inicio=1, fin=1


## count_rows_by_time

`count_rows_by_time` agrupa registros por periodicidad (`freq`, e.g. `'D'`, `'ME'`) y cuenta las filas por cada intervalo.

In [5]:
df_events = pd.DataFrame({
    "event_time": [
        "2024-01-01 08:00:00", "2024-01-01 12:00:00", "2024-01-01 18:00:00",
        "2024-01-02 09:00:00", "2024-01-15 14:00:00", "2024-02-01 10:00:00"
    ],
    "id": range(1, 7)
})

df_counts_daily = count_rows_by_time(df=df_events, col_time="event_time", freq="D")
display(df_counts_daily)

,event_time,n_rows
0,2024-01-01,3
1,2024-01-02,1
2,2024-01-15,1
3,2024-02-01,1


## count_constant_columns

`count_constant_columns` cuenta cuántas columnas tienen un único valor homogéneo (`nunique == 1`), permitiendo considerar o ignorar nulos mediante `dropna`.

In [6]:
df_const = pd.DataFrame({
    "const_num": [100, 100, 100],
    "const_nan": ["A", np.nan, "A"],
    "variable": [1, 2, 3]
})

print("Constantes con dropna=True :", count_constant_columns(df=df_const, dropna=True))
print("Constantes con dropna=False:", count_constant_columns(df=df_const, dropna=False))

Constantes con dropna=True : 2
Constantes con dropna=False: 1


## count_cells_with_missing

`count_cells_with_missing` suma todas las celdas no válidas (nulos estándar más cadenas vacías o espacios en blanco).

In [7]:
df_missing = pd.DataFrame({
    "num": [10.0, np.nan, 30.0, np.nan],       # 2 NaN
    "txt": ["Madrid", "", "Valencia", "   "],  # 2 cadenas vacías
    "ok":  [1, 2, 3, 4]                       # 0 nulos
})

print("Total de celdas ausentes en el DataFrame:", count_cells_with_missing(df=df_missing))

Total de celdas ausentes en el DataFrame: 4


## count_rows_with_missing

`count_rows_with_missing` calcula cuántas filas tienen al menos una celda ausente.

In [8]:
print("Total de filas con algún dato ausente:", count_rows_with_missing(df=df_missing))

Total de filas con algún dato ausente: 2


## count_columns_with_missing

`count_columns_with_missing` calcula cuántas columnas tienen al menos un dato ausente.

In [9]:
print("Total de columnas con algún dato ausente:", count_columns_with_missing(df=df_missing))

Total de columnas con algún dato ausente: 2


## count_missing_cells_by_column

`count_missing_cells_by_column` devuelve un `pd.Series` con la cantidad exacta de ausentes desglosada por cada columna.

In [10]:
series_missing = count_missing_cells_by_column(df=df_missing)
display(series_missing)

num    2
ok     0
txt    2
dtype: int64